# 02 · BM25 baseline (Prompt 5, owner: M1)

Before running:
1. **Accelerator: None** (CPU only, saves GPU quota). **Internet: On**.
2. **Add Input → Your Work → Datasets → `fever-corpus-v1`** (the Phase 1 output).

Expected run time: about 20-40 minutes (install, Lucene index of 308k sentences, 12 k1/b settings on
2k validation claims, 20k test claims, 30k train claims).

When it finishes: **Save Version → Save & Run All (Commit)**. Notebook `03_dense` adds this notebook's
output as an input (it needs `outputs/bm25/run_test.jsonl`).

In [1]:
import os, subprocess
REPO_URL = "https://github.com/group-mauve/claim-verification-pipeline.git"   # public repo
BRANCH = "main"
WORK = "/kaggle/working/project"

def git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    print((r.stdout + r.stderr).strip())
    if r.returncode != 0:
        raise RuntimeError("git failed (see message above). Is Internet on in the notebook settings?")

if not os.path.exists(WORK):
    git("clone", "--depth", "1", "--branch", BRANCH, REPO_URL, WORK)
else:
    git("-C", WORK, "pull", "origin", BRANCH)   # pick up teammates' latest pushes
os.chdir(WORK)
git("log", "--oneline", "-1")                    # which commit this run uses
print(os.getcwd(), sorted(os.listdir()))

Cloning into '/kaggle/working/project'...
2a4b86d Find Kaggle inputs through symlinks; list inputs when missing
/kaggle/working/project ['.git', '.gitignore', 'README.md', 'configs', 'data', 'notebooks', 'pytest.ini', 'requirements.txt', 'src', 'tests']


## Install Java 21 and Pyserini
Pyserini runs Lucene inside a Java virtual machine and needs Java 21. We try `apt-get` first and fall back to downloading a JDK with `install-jdk`. The JDK goes to `/tmp`, so it is not saved in the notebook output.

In [2]:
import os, subprocess, sys, glob

def java_home_from_apt():
    cands = sorted(glob.glob("/usr/lib/jvm/java-21-openjdk*"))
    return cands[0] if cands else None

home = java_home_from_apt()
if home is None:
    r = subprocess.run("apt-get update -qq && apt-get install -y -qq openjdk-21-jdk-headless", shell=True)
    home = java_home_from_apt()
if home is None:   # fallback: download a JDK 21 into /tmp
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "install-jdk"], check=True)
    import jdk
    home = jdk.install("21", path="/tmp/jdk")
os.environ["JAVA_HOME"] = home
os.environ["PATH"] = os.path.join(home, "bin") + os.pathsep + os.environ["PATH"]
print("JAVA_HOME =", home)
!java -version

JAVA_HOME = /usr/lib/jvm/java-21-openjdk-amd64
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)


In [3]:
!pip install -q pyserini rank_bm25 pyyaml pytest
!python -c "from pyserini.search.lucene import LuceneSearcher; print('pyserini OK')"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-sdk 1.42.1 requires opentelemetry-api==1.42.1, but you have opentelemetry-api 1.45.2 which is incompatible.
opentelemetry-semantic-conventions 0.63b1 requires opentelemetry-api==1.42.1, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
pyserini OK


## Unit tests

In [4]:
!python -m pytest -q -rs

.................s..........                                             [100%]
=========================== short test summary info ============================
SKIPPED [1] tests/test_metrics.py:125: clone sheffieldnlp/fever-scorer and add its src/ folder to PYTHONPATH
27 passed, 1 skipped in 4.14s


## Quick test: rank_bm25 on a 50k-sentence sample (no Java involved)
Checks the data and evaluation path in about a minute before the full run.

In [5]:
!python -m src.retrieval.bm25 --config configs/bm25.yaml --quick

06:38:36 | INFO | Corpus: /kaggle/input/datasets/atulckulkarni/fever-corpus-v1/fever_corpus_v1/corpus | splits: /kaggle/input/datasets/atulckulkarni/fever-corpus-v1/fever_corpus_v1/splits
06:38:37 | INFO | Corpus sentences: 308,328 | val claims: 2,000
06:38:37 | INFO | Quick test: 50,000 sentences, 300 val claims
06:38:59 | INFO | k1=0.6  b=0.4  val sentence recall@10 = 0.9030  (page recall@10 0.9634)
06:39:18 | INFO | k1=0.6  b=0.75 val sentence recall@10 = 0.8967  (page recall@10 0.9656)
06:39:38 | INFO | k1=0.6  b=0.9  val sentence recall@10 = 0.8852  (page recall@10 0.9624)
06:39:57 | INFO | k1=0.9  b=0.4  val sentence recall@10 = 0.9014  (page recall@10 0.9656)
06:40:18 | INFO | k1=0.9  b=0.75 val sentence recall@10 = 0.8894  (page recall@10 0.9666)
06:40:37 | INFO | k1=0.9  b=0.9  val sentence recall@10 = 0.8648  (page recall@10 0.9666)
06:40:58 | INFO | k1=1.2  b=0.4  val sentence recall@10 = 0.9010  (page recall@10 0.9677)
06:41:17 | INFO | k1=1.2  b=0.75 val sentence recall@10

## Full run: Pyserini index, k1/b tuning on validation, test metrics

In [6]:
!python -m src.retrieval.bm25 --config configs/bm25.yaml

06:43:02 | INFO | Corpus: /kaggle/input/datasets/atulckulkarni/fever-corpus-v1/fever_corpus_v1/corpus | splits: /kaggle/input/datasets/atulckulkarni/fever-corpus-v1/fever_corpus_v1/splits
06:43:03 | INFO | Corpus sentences: 308,328 | val claims: 2,000
06:43:06 | INFO | Wrote 308,328 documents to /kaggle/working/project/outputs/bm25/collection; indexing with Pyserini ...
2026-11-20 06:43:09,484 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:208) - Setting log level to INFO
2026-11-20 06:43:09,490 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:211) - ============ Loading Index Configuration ============
2026-11-20 06:43:09,490 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:212) - AbstractIndexer settings:
2026-11-20 06:43:09,493 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:213) -  + DocumentCollection path: /kaggle/working/project/outputs/bm25/collection
2026-11-20 06:43:09,493 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:214) -  + Co

In [7]:
import json, pandas as pd
m = json.load(open("outputs/bm25/metrics.json"))
print(f"best k1 = {m['best_k1']}, b = {m['best_b']} | test split {m['test_split']} | "
      f"{m['ms_per_query']} ms/query (single), {m['batch_ms_per_query']} ms/query (batch)")
print(pd.DataFrame({lv: m["test"][lv] for lv in ("page", "sentence")}).round(4))
pd.DataFrame(m["tuning_grid"])[["k1", "b", "sentence_recall@10", "page_recall@10", "sentence_mrr"]].round(4)

best k1 = 0.6, b = 0.4 | test split test_full | 3.96 ms/query (single), 2.57 ms/query (batch)
                       page  sentence
recall@5             0.8835    0.7573
recall@10            0.9292    0.8267
recall@20            0.9527    0.8808
mrr                  0.7973    0.7220
ndcg@10              0.8115    0.7152
evidence_set_hit@5      NaN    0.7853
evidence_set_hit@10     NaN    0.8472
evidence_set_hit@20     NaN    0.8918


,k1,b,sentence_recall@10,page_recall@10,sentence_mrr
0,0.6,0.40,0.8052,0.9131,0.6812
1,0.6,0.75,0.7870,0.9122,0.6629
2,0.6,0.90,0.7767,0.9110,0.6472
3,0.9,0.40,0.7993,0.9122,0.6708
4,0.9,0.75,0.7693,0.9100,0.6366
5,0.9,0.90,0.7558,0.9086,0.6158
6,1.2,0.40,0.7911,0.9106,0.6491
7,1.2,0.75,0.7564,0.9079,0.6094
8,1.2,0.90,0.7406,0.9029,0.5858
9,1.5,0.40,0.7848,0.9096,0.6317


In [8]:
!du -sh outputs/bm25/*

4.0K	outputs/bm25/best_params.json
4.0K	outputs/bm25/config_used.yaml
24M	outputs/bm25/lucene_index
12K	outputs/bm25/metrics.json
4.0K	outputs/bm25/run.log
81M	outputs/bm25/run_test.jsonl
25M	outputs/bm25/run_train.jsonl
8.2M	outputs/bm25/run_val.jsonl
